# Zipformer 30M + DeepFilterNet3 — benchmark tự động từ dataset
Chọn **T4 GPU → Run all**. Không cần chọn/upload audio.
Notebook tự tải 20 câu Việt FLEURS và 6 loại nhiễu DCASE: keyboard, gõ cửa, đóng cửa, chìa khóa, lật giấy, chuông điện thoại.
**7 điều kiện**: keyboard liên tục + 6 loại chèn theo sự kiện. Mỗi điều kiện đo ở SNR 0/5/10 dB.
Mặc định tạo **20 câu sạch + 420 bản trộn + 21 mẫu chỉ có nhiễu = 461 đầu vào / 922 kết quả ASR**.
Có benchmark latency lặp trên một mẫu tự chọn, bảng WER/CER riêng từng loại và chế độ nhiễu, audio đối chiếu và ZIP kết quả.
Giữ DF3 trong Python 3.12.3 riêng; CUDA phải được xác minh bằng profile node thực thi của chính Sherpa.

## 1. Kiểm tra GPU NVIDIA

In [ ]:
import sys, subprocess, platform
from pathlib import Path
assert platform.system() == 'Linux' and platform.machine() == 'x86_64', 'Cần Colab Linux x86_64'
assert sys.version_info[:2] in ((3, 10), (3, 11), (3, 12), (3, 13)), sys.version
subprocess.run(['nvidia-smi'], check=True)
Path('/content').mkdir(exist_ok=True)
print('Notebook Python:', sys.version)

Notebook Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]


## 2. Cài CUDA wheel cho Sherpa
Dùng wheel CUDA 12/cuDNN 9 được liệt kê tại [k2-fsa](https://k2-fsa.github.io/sherpa/onnx/cuda.html).
Không cần cài Python `onnxruntime-gpu`: Sherpa dùng runtime đóng gói riêng.
Không ép hạ `huggingface-hub`. `nvidia-ml-py` cung cấp module `pynvml`.
Worker mới nhận `LD_LIBRARY_PATH` trước khi khởi động; không sửa symlink trong site-packages.

In [ ]:
import sys, subprocess
SHERPA_VERSION = '1.13.8+cuda12.cudnn9.onnxruntime1.28.2'
def pip_install(*packages):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *packages], check=True)

pip_install('numpy', 'scipy', 'soundfile', 'sentencepiece', 'huggingface-hub', 'nvidia-ml-py')
pip_install('nvidia-cudnn-cu12>=9,<10', 'nvidia-cublas-cu12', 'nvidia-cufft-cu12',
            'nvidia-curand-cu12', 'nvidia-cuda-nvrtc-cu12', 'nvidia-cuda-runtime-cu12',
            'nvidia-cusparse-cu12', 'nvidia-cusolver-cu12', 'nvidia-nvjitlink-cu12')
pip_install('--force-reinstall', '--no-deps', f'sherpa-onnx=={SHERPA_VERSION}',
            '-f', 'https://k2-fsa.github.io/sherpa/onnx/cuda.html')

## 3. Tải checkpoint ONNX FP32 và tokens

In [ ]:
from huggingface_hub import snapshot_download
from pathlib import Path

MODEL_ID = "hynt/Zipformer-30M-RNNT-6000h"

model_dir = Path(
    snapshot_download(
        repo_id=MODEL_ID,
        revision="ad6f873b5f5b6ed083821253a88d81afed2505c4",
        allow_patterns=[
            "encoder-epoch-20-avg-10.onnx",
            "decoder-epoch-20-avg-10.onnx",
            "joiner-epoch-20-avg-10.onnx",
            "bpe.model",
        ],
    )
)

print("Model directory:", model_dir)

for name in [
    "encoder-epoch-20-avg-10.onnx",
    "decoder-epoch-20-avg-10.onnx",
    "joiner-epoch-20-avg-10.onnx",
    "bpe.model",
]:
    assert (model_dir / name).exists(), name
    print("OK:", name)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Model directory: /root/.cache/huggingface/hub/models--hynt--Zipformer-30M-RNNT-6000h/snapshots/ad6f873b5f5b6ed083821253a88d81afed2505c4
OK: encoder-epoch-20-avg-10.onnx
OK: decoder-epoch-20-avg-10.onnx
OK: joiner-epoch-20-avg-10.onnx
OK: bpe.model


In [ ]:
import urllib.request
import tarfile
from pathlib import Path

url = (
    "https://github.com/k2-fsa/sherpa-onnx/releases/"
    "download/asr-models/"
    "sherpa-onnx-zipformer-vi-30M-int8-2026-02-09.tar.bz2"
)

archive = Path("/content/zipformer_tokens.tar.bz2")

if not archive.exists():
    print("Downloading official sherpa-onnx package...")
    urllib.request.urlretrieve(url, archive)

with tarfile.open(archive, "r:bz2") as tar:
    token_files = [
        member for member in tar.getmembers()
        if Path(member.name).name == "tokens.txt"
        and member.isfile()
    ]

    if len(token_files) != 1:
        raise RuntimeError("Không tìm thấy duy nhất một tokens.txt")

    source = tar.extractfile(token_files[0])
    if source is None:
        raise RuntimeError("Không đọc được tokens.txt")

    token_data = source.read()

tokens_path = Path("/content/zipformer_tokens.txt")
tokens_path.write_bytes(token_data)

print("Tokens file:", tokens_path)
print("Size:", tokens_path.stat().st_size, "bytes")

Tokens file: /content/zipformer_tokens.txt
Size: 23238 bytes


## 4. Cài DeepFilterNet3 trong Python 3.12.3 riêng
Giữ cách build pyDF bằng maturin và PyTorch/torchaudio 2.5.1 CUDA 12.1.
Không thay Python của kernel Colab. Model DF3 phải nằm trên CUDA; STFT/synthesis vẫn có phần CPU.

In [ ]:
import os
import sys
import subprocess
import shutil
from pathlib import Path

def run_cmd(args, env=None):
    args = [str(x) for x in args]
    print("\n+", " ".join(args), flush=True)

    subprocess.run(
        args,
        check=True,
        env=env
    )

run_cmd([
    sys.executable, "-m", "pip", "install", "-q",
    "uv",
    "maturin>=1.5,<2"
])

run_cmd([
    "uv", "python", "install", "3.12.3"
])

DF_ENV = Path("/content/df3-py312")
DF_PY = DF_ENV / "bin" / "python"

if not DF_PY.exists():
    run_cmd([
        "uv", "venv",
        "--seed",
        "--python", "3.12.3",
        str(DF_ENV)
    ])

run_cmd([
    DF_PY, "-c",
    "import sys; "
    "print('Python:', sys.version); "
    "assert sys.version_info[:3] == (3, 12, 3)"
])

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),

    "torch==2.5.1",
    "torchaudio==2.5.1",

    "--index-url",
    "https://download.pytorch.org/whl/cu121"
])

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),

    "numpy>=1.24,<2",
    "scipy",
    "soundfile>=0.10,<0.13",
    "loguru",
    "appdirs",
    "requests",
    "sympy",
    "packaging>=23,<25",
    "poetry-core>=1"
])

run_cmd([
    DF_PY, "-c",
    "import torch, torchaudio; "
    "print('PyTorch:', torch.__version__); "
    "print('Torchaudio:', torchaudio.__version__); "
    "print('CUDA available:', torch.cuda.is_available()); "
    "print('CUDA version:', torch.version.cuda); "
    "assert torch.cuda.is_available(), 'CUDA is unavailable'"
])

rust_bin = Path.home() / ".cargo" / "bin"

build_env = os.environ.copy()

build_env["PATH"] = (
    str(rust_bin)
    + os.pathsep
    + build_env.get("PATH", "")
)

if shutil.which("cargo", path=build_env["PATH"]) is None:
    run_cmd([
        "bash", "-lc",
        'curl --proto "=https" --tlsv1.2 -sSf '
        'https://sh.rustup.rs | sh -s -- -y --profile minimal'
    ])

if (rust_bin / "rustup").exists():
    run_cmd([
        rust_bin / "rustup",
        "update",
        "stable"
    ], env=build_env)

run_cmd([
    "cargo", "--version"
], env=build_env)

repo = Path("/content/DeepFilterNet-src")

if not (repo / "pyDF" / "Cargo.toml").is_file():
    run_cmd([
        "git", "clone",
        "--depth", "1",
        "https://github.com/Rikorose/DeepFilterNet.git",
        str(repo)
    ])

wheels = Path("/content/df3-wheels-py312")
wheels.mkdir(parents=True, exist_ok=True)

libdf_wheels = sorted(
    wheels.glob("*deepfilterlib*cp312*.whl")
)

if not libdf_wheels:

    run_cmd([
        sys.executable,
        "-m", "maturin",
        "build",
        "--release",
        "--interpreter", str(DF_PY),
        "--manifest-path", str(repo / "pyDF" / "Cargo.toml"),
        "--out", str(wheels)
    ], env=build_env)

    libdf_wheels = sorted(
        wheels.glob("*deepfilterlib*cp312*.whl")
    )

if not libdf_wheels:
    libdf_wheels = sorted(
        wheels.glob("*deepfilterlib*.whl")
    )

assert libdf_wheels, (
    f"No deepfilterlib wheel found: "
    f"{list(wheels.glob('*.whl'))}"
)

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),
    "--no-deps",
    str(libdf_wheels[-1])
])

run_cmd([
    "uv", "pip", "install",
    "--python", str(DF_PY),
    "--no-deps",
    str(repo / "DeepFilterNet")
])

verify_code = """
import sys
import torch
import torchaudio
import df
import libdf

from df.enhance import init_df, enhance

print("=" * 55)
print("DeepFilterNet3 Environment Verification")
print("=" * 55)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchaudio:", torchaudio.__version__)
print("CUDA version:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

assert torch.cuda.is_available(), (
    "DeepFilterNet3 requires CUDA"
)

print("GPU:", torch.cuda.get_device_name(0))
print("DeepFilterNet import: OK")
print("deepfilterlib import: OK")

print("=" * 55)
print("DeepFilterNet3 installation successful!")
print("=" * 55)
"""

run_cmd([
    DF_PY,
    "-c",
    verify_code
])

print("\nDeepFilterNet3 isolated Python 3.12.3 installation complete.")


+ /usr/bin/python3 -m pip install -q uv maturin>=1.5,<2

+ uv python install 3.12.3

+ uv venv --seed --python 3.12.3 /content/df3-py312

+ /content/df3-py312/bin/python -c import sys; print('Python:', sys.version); assert sys.version_info[:3] == (3, 12, 3)

+ uv pip install --python /content/df3-py312/bin/python torch==2.5.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121

+ uv pip install --python /content/df3-py312/bin/python numpy>=1.24,<2 scipy soundfile>=0.10,<0.13 loguru appdirs requests sympy packaging>=23,<25 poetry-core>=1

+ /content/df3-py312/bin/python -c import torch, torchaudio; print('PyTorch:', torch.__version__); print('Torchaudio:', torchaudio.__version__); print('CUDA available:', torch.cuda.is_available()); print('CUDA version:', torch.version.cuda); assert torch.cuda.is_available(), 'CUDA is unavailable'

+ bash -lc curl --proto "=https" --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y --profile minimal

+ /root/.cargo/bin/rustup update stab

## 5. Khởi tạo mã worker và cấu hình dataset
Nguồn: [FLEURS Vietnamese test](https://huggingface.co/datasets/google/fleurs), CC BY 4.0;
[DCASE 2016 Task 2](https://archive.org/details/dcase2016_task2_train_dev), CC BY 3.0 theo metadata archive.
DCASE khoảng 125 MB có retry, máy chủ dự phòng và checksum. FLEURS đọc streaming đến khi đủ câu.
Đổi `N_SPEECH` để thử nhanh hơn. Speech FLEURS gốc 16 kHz: resample lên 48 kHz cho DF3 không khôi phục dải tần đã mất.

In [ ]:
ASR_WORKER_SOURCE = r'''
import argparse, csv, ctypes, gc, json, os, time, threading
from pathlib import Path
from collections import Counter
from importlib.metadata import version
import numpy as np
import soundfile as sf
import pynvml
import sherpa_onnx


def inspect_cuda_profiles(paths):
    counts = Counter()
    for path in paths:
        events = json.loads(Path(path).read_text())
        for event in events:
            provider = event.get('args', {}).get('provider')
            if event.get('cat') == 'Node' and provider:
                counts[provider] += 1
    if counts['CUDAExecutionProvider'] == 0:
        raise RuntimeError('Không có node CUDA thực thi trong profile Sherpa; dừng benchmark.')
    return dict(counts)


def main():
    p = argparse.ArgumentParser()
    p.add_argument('--mode', choices=['probe', 'benchmark'], required=True)
    p.add_argument('--model-dir', required=True)
    p.add_argument('--tokens', required=True)
    p.add_argument('--profile-dir', required=True)
    a = p.parse_args()
    model_dir = Path(a.model_dir)
    profile_dir = Path(a.profile_dir)
    if a.mode == 'probe':
        config = profile_dir / 'provider.conf'
        config.write_text(f'ProfilingFilePrefix={profile_dir}/sherpa\nLogSeverityLevel=1\n')
        provider = f'cuda:{config}'
    else:
        evidence = inspect_cuda_profiles(sorted(profile_dir.glob('sherpa*.json')))
        provider = 'cuda'

    # CUDA_VISIBLE_DEVICES is set to physical GPU 0 by the notebook.
    pynvml.nvmlInit()
    handle = pynvml.nvmlDeviceGetHandleByIndex(0)
    gpu_name = pynvml.nvmlDeviceGetName(handle)
    if isinstance(gpu_name, bytes):
        gpu_name = gpu_name.decode()
    def memory():
        total = pynvml.nvmlDeviceGetMemoryInfo(handle).used / 2**20
        processes = pynvml.nvmlDeviceGetComputeRunningProcesses(handle)
        used = [x.usedGpuMemory for x in processes if x.pid == os.getpid()]
        pid_mib = sum(used) / 2**20 if used and all(x is not None and x != (1 << 64) - 1 for x in used) else None
        return {'total_mib': total, 'process_mib': pid_mib}

    before = memory()
    start = time.perf_counter()
    recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
        tokens=a.tokens, encoder=str(model_dir/'encoder-epoch-20-avg-10.onnx'),
        decoder=str(model_dir/'decoder-epoch-20-avg-10.onnx'),
        joiner=str(model_dir/'joiner-epoch-20-avg-10.onnx'),
        num_threads=2, sample_rate=16000, feature_dim=80,
        decoding_method='greedy_search', provider=provider)
    # Only the CUDA runtime is loaded; do not create a Torch context for measurement.
    cudart = ctypes.CDLL('libcudart.so.12')
    cudart.cudaDeviceSynchronize.restype = ctypes.c_int
    def synchronize():
        status = cudart.cudaDeviceSynchronize()
        if status:
            raise RuntimeError(f'cudaDeviceSynchronize failed: {status}')
    synchronize()
    load_seconds = time.perf_counter() - start
    after = memory()
    print('Sherpa:', version('sherpa-onnx'), '| GPU:', gpu_name, flush=True)
    print('Memory before/after load:', before, after, flush=True)

    def recognize_once(signal):
        synchronize()
        start = time.perf_counter()
        stream = recognizer.create_stream()
        stream.accept_waveform(16000, signal)
        recognizer.decode_stream(stream)
        text = stream.result.text
        synchronize()
        return text, time.perf_counter() - start

    if a.mode == 'probe':
        signal, sr = sf.read('/content/asr_test_16k.wav', dtype='float32')
        assert sr == 16000
        print('Probe:', recognize_once(signal), flush=True)
        del recognizer
        gc.collect()
        return  # Process exit also destroys ORT sessions and flushes profiling.

    class GPUMonitor:
        def __init__(self):
            self.samples = []
            self.errors = []
            self.stop = threading.Event()
        def __enter__(self):
            self.samples.append(memory())
            self.thread = threading.Thread(target=self.poll, daemon=True)
            self.thread.start()
            return self
        def poll(self):
            while not self.stop.wait(0.02):
                try:
                    self.samples.append(memory())
                except Exception as error:
                    self.errors.append(str(error))
                    return
        def __exit__(self, *args):
            self.stop.set()
            self.thread.join()
            self.samples.append(memory())

    audios = {}
    for name, file in [('raw', 'asr_test_16k.wav'), ('df3', 'asr_test_df3_16k.wav')]:
        signal, sr = sf.read('/content/'+file, dtype='float32')
        assert sr == 16000 and signal.ndim == 1 and signal.size > 0
        assert np.isfinite(signal).all()
        audios[name] = signal
    assert len(audios['raw']) == len(audios['df3']), 'Raw/DF3 khác độ dài'
    results = {name: [] for name in audios}
    n_warmup, n_runs = 5, 20
    baseline = memory()
    with GPUMonitor() as monitor:
        # Warm both variants with monitoring already enabled.
        for _ in range(n_warmup):
            for signal in audios.values():
                recognize_once(signal)
        # Alternate order to reduce systematic raw-first drift; retain all runs.
        for index in range(n_runs):
            order = ('raw', 'df3') if index % 2 == 0 else ('df3', 'raw')
            for name in order:
                results[name].append(recognize_once(audios[name]))
    df_metrics = json.loads(Path('/content/df3_metrics.json').read_text())
    pid_samples = [s['process_mib'] for s in monitor.samples if s['process_mib'] is not None]
    for name, runs in results.items():
        latencies = [r[1] for r in runs]
        texts = [r[0] for r in runs]
        seconds = len(audios[name]) / 16000
        median = float(np.median(latencies))
        p95 = float(np.percentile(latencies, 95))
        report = {
            'model': 'hynt/Zipformer-30M-RNNT-6000h', 'model_revision': model_dir.name,
            'audio_variant': name, 'runtime': 'sherpa-onnx ONNX FP32',
            'sherpa_version': version('sherpa-onnx'), 'gpu': gpu_name,
            'driver': pynvml.nvmlSystemGetDriverVersion(), 'requested_provider': 'cuda',
            'cuda_probe_executed_node_counts': evidence,
            'decoding_method': 'greedy_search', 'num_threads': 2,
            'model_load_seconds': load_seconds,
            'warmup_runs': n_warmup, 'measured_runs': n_runs, 'audio_seconds': seconds,
            'median_latency_seconds': median, 'p95_latency_seconds': p95,
            'median_rtf': median/seconds, 'p95_rtf': p95/seconds,
            'gpu_memory_before_load': before, 'gpu_memory_after_load': after,
            'gpu_memory_baseline': baseline,
            'gpu_memory_peak_total_mib': max(s['total_mib'] for s in monitor.samples),
            'gpu_memory_peak_process_mib': max(pid_samples) if pid_samples else None,
            'memory_monitor_errors': monitor.errors,
            'text': texts[0], 'transcript_consistent': len(set(texts)) == 1,
            'transcripts': texts, 'latencies_seconds': latencies,
            'audio_peak': float(np.max(np.abs(audios[name]))),
            'audio_fraction_above_one': float(np.mean(np.abs(audios[name]) > 1)),
            'deepfilternet3': df_metrics if name == 'df3' else None,
            'approx_pipeline_median_seconds': median + df_metrics['median_enhancement_seconds'] + df_metrics['single_resample_seconds'] if name == 'df3' else None,
            'notes': 'Latency: stream + waveform + decode + text + CUDA sync; excludes model load/I/O. NVML 20ms sampled peak includes warmup; PID memory includes CUDA context/allocator. Pipeline sum is an estimate across separate warm workers, not measured end-to-end or a median of sums.'}
        # Ensure old drivers returning bytes still produce valid JSON.
        if isinstance(report['driver'], bytes):
            report['driver'] = report['driver'].decode()
        prefix = Path(f'/content/zipformer30m_{name}_benchmark')
        prefix.with_suffix('.json').write_text(json.dumps(report, indent=2, ensure_ascii=False))
        with prefix.with_suffix('.csv').open('w', newline='') as f:
            writer = csv.DictWriter(f, fieldnames=['run', 'latency_seconds', 'rtf'])
            writer.writeheader()
            writer.writerows({'run': i+1, 'latency_seconds': t, 'rtf': t/seconds}
                             for i, t in enumerate(latencies))
        print(f'{name}: median={median:.6f}s; p95={p95:.6f}s; RTF={median/seconds:.6f}', flush=True)
        print('Transcript:', texts[0], '| consistent:', report['transcript_consistent'], flush=True)
    pynvml.nvmlShutdown()


if __name__ == '__main__':
    main()
'''


import os, sys, subprocess, site, importlib.util
from pathlib import Path
asr_worker = Path('/content/asr_worker.py')
asr_worker.write_text(ASR_WORKER_SOURCE)
spec = importlib.util.find_spec('sherpa_onnx')
assert spec and spec.origin
sherpa_lib_dir = Path(spec.origin).parent / 'lib'
assert (sherpa_lib_dir/'libonnxruntime_providers_cuda.so').exists(), 'Wheel thiếu CUDA provider .so'
lib_dirs = [sherpa_lib_dir]
for root in site.getsitepackages() + [site.getusersitepackages()]:
    lib_dirs.extend(sorted(Path(root).glob('nvidia/*/lib')))
asr_env = os.environ.copy()
asr_env['CUDA_VISIBLE_DEVICES'] = '0'
asr_env['LD_LIBRARY_PATH'] = os.pathsep.join(str(p) for p in lib_dirs) + os.pathsep + asr_env.get('LD_LIBRARY_PATH', '')
asr_env['OMP_NUM_THREADS'] = '2'
# ldd inspects the actual shared library, not sherpa_onnx/__init__.py.
linked = subprocess.run(['ldd', str(sherpa_lib_dir/'libonnxruntime_providers_cuda.so')],
                        env=asr_env, capture_output=True, text=True, check=True)
print(linked.stdout)
assert 'not found' not in linked.stdout, 'Thiếu CUDA/cuDNN .so; xem ldd ở trên'

	linux-vdso.so.1 (0x00007ffe60f71000)
	libcublasLt.so.12 => /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublasLt.so.12 (0x00007879cee00000)
	libcublas.so.12 => /usr/local/lib/python3.13/dist-packages/nvidia/cublas/lib/libcublas.so.12 (0x00007879c8600000)
	libcurand.so.10 => /usr/local/lib/python3.13/dist-packages/nvidia/cu13/lib/libcurand.so.10 (0x00007879bfe00000)
	libcudart.so.12 => /usr/local/lib/python3.13/dist-packages/nvidia/cuda_runtime/lib/libcudart.so.12 (0x00007879bfa00000)
	libcuda.so.1 => /usr/lib64-nvidia/libcuda.so.1 (0x00007879b9c00000)
	libdl.so.2 => /lib/x86_64-linux-gnu/libdl.so.2 (0x0000787a01258000)
	librt.so.1 => /lib/x86_64-linux-gnu/librt.so.1 (0x0000787a01253000)
	libpthread.so.0 => /lib/x86_64-linux-gnu/libpthread.so.0 (0x0000787a0124e000)
	libstdc++.so.6 => /lib/x86_64-linux-gnu/libstdc++.so.6 (0x00007879b9982000)
	libm.so.6 => /lib/x86_64-linux-gnu/libm.so.6 (0x0000787a01163000)
	libgcc_s.so.1 => /lib/x86_64-linux-gnu/libgcc_s.so.1 (0x0000787

In [ ]:
QUALITY_WORKER_SOURCE = r'''
"""Vietnamese speech benchmark with continuous and intermittent office noise; also embedded in the Colab notebook."""
import argparse
import csv
import hashlib
import http.client
import io
import json
import math
import os
from pathlib import Path
import tarfile
import time
import unicodedata
import urllib.request
import urllib.error
import zipfile

import numpy as np
import soundfile as sf
from scipy.signal import resample_poly

FLEURS_REVISION = '70bb2e84b976b7e960aa89f1c648e09c59f894dd'
FLEURS_BASE = f'https://huggingface.co/datasets/google/fleurs/resolve/{FLEURS_REVISION}/data/vi_vn'
DCASE_URL = 'https://archive.org/download/dcase2016_task2_train_dev/dcase2016_task2_train_dev.zip'
DCASE_MD5 = '4e1b5e8887159193e8624dec801eb9e7'
# Storage hosts returned by archive.org/metadata/dcase2016_task2_train_dev.
DCASE_MIRRORS = [
    f'https://{host}/17/items/dcase2016_task2_train_dev/dcase2016_task2_train_dev.zip'
    for host in ('ia801900.us.archive.org', 'ia601900.us.archive.org')
]


NOISE_PREFIXES = {'keyboard': 'keyboard', 'knock': 'knock', 'doorslam': 'doorslam',
                  'keys': 'keysDrop', 'pageturn': 'pageturn', 'phone': 'phone'}
NOISE_CONDITIONS = [('keyboard', 'continuous')] + [(name, 'events') for name in NOISE_PREFIXES]


def normalize_text(text):
    text = unicodedata.normalize('NFC', text).casefold()
    return ' '.join(''.join(c if c.isalnum() or c.isspace() else ' ' for c in text).split())


def edit_distance(reference, hypothesis):
    previous = list(range(len(hypothesis) + 1))
    for i, a in enumerate(reference, 1):
        current = [i]
        for j, b in enumerate(hypothesis, 1):
            current.append(min(current[-1] + 1, previous[j] + 1, previous[j-1] + (a != b)))
        previous = current
    return previous[-1]


def error_counts(reference, hypothesis):
    ref, hyp = normalize_text(reference), normalize_text(hypothesis)
    words, chars = ref.split(), ref.replace(' ', '')
    word_errors = edit_distance(words, hyp.split())
    char_errors = edit_distance(chars, hyp.replace(' ', ''))
    return dict(reference_normalized=ref, hypothesis_normalized=hyp,
                word_errors=word_errors, reference_words=len(words),
                char_errors=char_errors, reference_chars=len(chars),
                wer=word_errors/len(words) if words else None,
                cer=char_errors/len(chars) if chars else None, has_text=bool(hyp))


def mix_at_snr(speech, noise, snr_db, *, active_mask=None, speech_mask=None):
    speech, noise = np.asarray(speech, dtype=np.float64), np.asarray(noise, dtype=np.float64)
    if speech.ndim != 1 or speech.shape != noise.shape or not speech.size:
        raise ValueError('Speech/noise must be nonempty mono arrays of identical length')
    if not np.isfinite(speech).all() or not np.isfinite(noise).all() or not math.isfinite(snr_db):
        raise ValueError('Nonfinite audio or SNR')
    active_mask = np.ones(len(noise), dtype=bool) if active_mask is None else np.asarray(active_mask, dtype=bool)
    speech_mask = np.ones(len(speech), dtype=bool) if speech_mask is None else np.asarray(speech_mask, dtype=bool)
    if (active_mask.shape != speech.shape or speech_mask.shape != speech.shape
            or not active_mask.any() or not speech_mask.any()):
        raise ValueError('Invalid or empty activity mask')
    speech_rms = np.sqrt(np.mean(speech[speech_mask]**2))
    noise_rms = np.sqrt(np.mean(noise[active_mask]**2))
    if min(speech_rms, noise_rms) < 1e-10:
        raise ValueError('Cannot mix silent speech/noise at a defined SNR')
    scaled_noise = noise * (speech_rms / noise_rms) * 10**(-snr_db/20)
    mixed = speech + scaled_noise
    gain = min(1., .95/max(float(np.max(np.abs(mixed))), 1e-10))
    target = speech * gain
    mixed = mixed * gain
    actual_snr = 10*np.log10(np.mean(target[speech_mask]**2)/np.mean((scaled_noise[active_mask]*gain)**2))
    return mixed.astype(np.float32), target.astype(np.float32), float(actual_snr), gain


def summarize(rows):
    groups = {}
    for row in rows:
        groups.setdefault((row['kind'], row.get('noise_type'), row.get('noise_style'),
                           row['snr_db'], row['branch']), []).append(row)
    result = []
    for (kind, noise_type, noise_style, snr_db, branch), items in groups.items():
        words, chars = sum(x['reference_words'] for x in items), sum(x['reference_chars'] for x in items)
        seconds = sum(x['audio_seconds'] for x in items)
        group = dict(kind=kind, noise_type=noise_type, noise_style=noise_style,
                     snr_db=snr_db, branch=branch, clips=len(items),
                     reference_words=words, reference_chars=chars,
                     wer=sum(x['word_errors'] for x in items)/words if words else None,
                     cer=sum(x['char_errors'] for x in items)/chars if chars else None,
                     nonempty_transcript_rate=sum(x['has_text'] for x in items)/len(items),
                     median_asr_seconds=float(np.median([x['asr_seconds'] for x in items])),
                     aggregate_asr_rtf=sum(x['asr_seconds'] for x in items)/seconds)
        if branch == 'df3':
            estimates = [x['asr_seconds'] + x['enhancement_seconds'] + x['resample_seconds'] for x in items]
            group['median_pipeline_estimate_seconds'] = float(np.median(estimates))
        result.append(group)
    return result


def save_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')


def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def read_audio(source, target_sr=48000):
    wave, sr = sf.read(source, dtype='float32', always_2d=True)
    wave = wave.mean(axis=1)
    if wave.size == 0 or not np.isfinite(wave).all():
        raise ValueError('Empty or nonfinite audio')
    if sr != target_sr:
        divisor = math.gcd(sr, target_sr)
        wave = resample_poly(wave, target_sr//divisor, sr//divisor).astype(np.float32)
    return wave, sr


def download(url, path, *, fallback_urls=(), expected_md5=None, attempts=3):
    path = Path(path)
    if attempts < 1:
        raise ValueError('attempts must be positive')
    def checksum_ok(candidate):
        if expected_md5 is None:
            return True
        digest = hashlib.md5()
        with candidate.open('rb') as f:
            while chunk := f.read(1024*1024):
                digest.update(chunk)
        return digest.hexdigest() == expected_md5
    if path.exists() and checksum_ok(path):
        return path
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.part')
    urls = list(dict.fromkeys([url, *fallback_urls]))
    last_error = None
    for attempt in range(attempts):
        for source_url in urls:
            print(f'Downloading ({attempt+1}/{attempts}): {source_url}', flush=True)
            try:
                request = urllib.request.Request(source_url, headers={'User-Agent': 'ZipformerBenchmark/1.0'})
                with urllib.request.urlopen(request, timeout=60) as source, temporary.open('wb') as destination:
                    expected_size = source.headers.get('Content-Length')
                    received = 0
                    while chunk := source.read(1024*1024):
                        destination.write(chunk)
                        received += len(chunk)
                if expected_size is not None and received != int(expected_size):
                    raise OSError(f'Incomplete download: {received}/{expected_size} bytes')
                if not checksum_ok(temporary):
                    raise OSError('Downloaded file checksum mismatch')
                temporary.replace(path)
                return path
            except (urllib.error.URLError, OSError, http.client.HTTPException) as error:
                last_error = error
                temporary.unlink(missing_ok=True)
                print(f'Download failed: {error}', flush=True)
        if attempt + 1 < attempts:
            time.sleep(2**attempt)
    raise RuntimeError(f'Cannot download {url} after {attempts} attempts across {len(urls)} URLs. '
                       f'Last error: {last_error}. You may place the original file at {path}; '
                       'it will be reused only if its checksum is valid.') from last_error


def noise_segment(wave, length, rng):
    offset = int(rng.integers(len(wave)))
    # Circular crop; reuses short isolated keyboard recordings without introducing zero pads.
    segment = wave[(offset + np.arange(length)) % len(wave)]
    if np.sqrt(np.mean(segment.astype(float)**2)) < 1e-9:
        raise ValueError('Selected noise segment is silent')
    return segment, offset


def frame_activity(wave, sr, threshold_db=-35):
    """20ms RMS gate, an energy proxy rather than a speech VAD."""
    wave = np.asarray(wave, dtype=float)
    if wave.ndim != 1 or not wave.size or not np.isfinite(wave).all() or sr < 1:
        raise ValueError('Invalid waveform/sample rate')
    hop = max(1, round(.02*sr))
    padded = np.pad(wave, (0, (-len(wave)) % hop))
    rms = np.sqrt(np.mean(padded.reshape(-1, hop)**2, axis=1))
    active = rms > max(float(rms.max())*10**(threshold_db/20), 1e-10)
    return np.repeat(active, hop)[:len(wave)]


def activity_intervals(mask, sr):
    edges = np.diff(np.r_[False, mask, False].astype(np.int8))
    return [[int(start)/sr, int(stop)/sr] for start, stop in
            zip(np.flatnonzero(edges == 1), np.flatnonzero(edges == -1))]


def build_noise_track(bank, length, sr, rng, noise_type, style, speech_mask):
    if not bank or style not in ('events', 'continuous') or length < 1:
        raise ValueError('Invalid noise bank, style, or track length')
    speech_mask = np.asarray(speech_mask, dtype=bool)
    if speech_mask.shape != (length,):
        raise ValueError('Speech activity length mismatch')
    if style == 'continuous':
        source = bank[int(rng.integers(len(bank)))]
        track, offset = noise_segment(source['wave'], length, rng)
        return track, frame_activity(track, sr, -45), [dict(
            start_sample=0, end_sample=length, source_file=source['file'], source_offset=offset,
            onset_region='unconstrained', requested_onset='unconstrained',
            speech_overlap_fraction=float(speech_mask.mean()))]
    track = np.zeros(length, dtype=np.float32)
    events = []
    gap, margin, hop = round(.3*sr), round(.1*sr), max(1, round(.02*sr))
    count = int(rng.integers(2, 4)) if noise_type == 'keyboard' else int(rng.integers(1, 4))
    if noise_type == 'phone':
        count = 1
    first_region = int(rng.integers(2))
    for index in range(count):
        source = bank[int(rng.integers(len(bank)))]
        offset = 0
        if noise_type == 'keyboard':
            size = min(round(float(rng.uniform(.5, 2))*sr), length-2*margin-gap)
            if size < 1:
                break
            event_wave, offset = noise_segment(source['wave'], size, rng)
            # Fade only synthetic keyboard burst boundaries, not transient source events.
            event_wave = event_wave.copy()
            fade = min(round(.005*sr), size//2)
            if fade:
                event_wave[:fade] *= np.linspace(0, 1, fade)
                event_wave[-fade:] *= np.linspace(1, 0, fade)
        elif noise_type == 'phone':
            # Preserve recorded rings; separate repeated rings by a natural quiet gap.
            pieces = []
            for ring in range(int(rng.integers(2, 4))):
                if ring:
                    pieces.append(np.zeros(round(.6*sr), dtype=np.float32))
                pieces.append(source['wave'])
            event_wave = np.concatenate(pieces)
            if len(event_wave) > min(5*sr, length-2*margin-gap):
                event_wave = source['wave']  # Use one complete ring instead of cutting its tail.
        else:
            event_wave = source['wave']  # Keep the full impact/page-turn and its decay.
        size = len(event_wave)
        if size > length-2*margin-gap:
            continue
        candidates = np.arange(margin, length-margin-size+1, hop)
        for previous in events:
            candidates = candidates[(candidates+size+gap <= previous['start_sample']) |
                                    (candidates >= previous['end_sample']+gap)]
        if not candidates.size:
            continue
        requested = 'speech' if (index+first_region) % 2 == 0 else 'pause'
        preferred = candidates[speech_mask[candidates] == (requested == 'speech')]
        onset_region = requested if preferred.size else 'unconstrained'
        start = int(rng.choice(preferred if preferred.size else candidates))
        stop = start+size
        track[start:stop] = event_wave
        event_mask = frame_activity(event_wave, sr, -45)
        events.append(dict(start_sample=start, end_sample=stop, source_file=source['file'],
                           source_offset=offset, requested_onset=requested, onset_region=onset_region,
                           speech_overlap_fraction=float(speech_mask[start:stop][event_mask].mean())))
    if not events:
        raise ValueError(f'No complete {noise_type} event fits this utterance')
    return track, frame_activity(track, sr, -45), sorted(events, key=lambda e: e['start_sample'])


def prepare(root, count=20, seed=20261019):
    if count < 1:
        raise ValueError('count must be positive')
    root = Path(root)
    cache = root/'cache'
    cache.mkdir(parents=True, exist_ok=True)
    for folder in ('input_48k', 'raw_16k', 'df3_16k', 'noise_sources'):
        (root/folder).mkdir(exist_ok=True)
    rng = np.random.default_rng(seed)
    archive = download(DCASE_URL, cache/'dcase2016_task2_train_dev.zip',
                       fallback_urls=DCASE_MIRRORS, expected_md5=DCASE_MD5)
    banks = {name: [] for name in NOISE_PREFIXES}
    with zipfile.ZipFile(archive) as z:
        members = sorted(n for n in z.namelist() if n.lower().endswith('.wav') and 'train/' in n)
        for noise_type, prefix in NOISE_PREFIXES.items():
            selected = [n for n in members if Path(n).stem.lower().startswith(prefix.lower())]
            for index, member in enumerate(selected):
                payload = z.read(member)
                wave, native_sr = read_audio(io.BytesIO(payload))
                if np.sqrt(np.mean(wave.astype(float)**2)) < 1e-9:
                    continue
                name = f'{noise_type}_{index:02d}.wav'
                sf.write(root/'noise_sources'/name, wave, 48000, subtype='FLOAT')
                banks[noise_type].append(dict(wave=wave, member=member, native_sr=native_sr,
                      sha256=hashlib.sha256(payload).hexdigest(), file=name, noise_type=noise_type))
            if not banks[noise_type]:
                raise RuntimeError(f'No valid {noise_type} recordings in DCASE archive')
    tsv = download(FLEURS_BASE+'/test.tsv', cache/'fleurs_vi_test.tsv')
    references = {}
    with tsv.open(encoding='utf-8') as f:
        for fields in csv.reader(f, delimiter='\t', quoting=csv.QUOTE_NONE):
            if len(fields) < 7:
                raise ValueError('Unexpected FLEURS TSV schema')
            references[fields[1]] = dict(prompt_id=fields[0], text=fields[2], gender=fields[6])
    speech = []
    used_prompts = set()
    # Stream only the required prefix of the 544 MB archive instead of downloading all speech.
    print('Streaming FLEURS Vietnamese test audio...', flush=True)
    with urllib.request.urlopen(FLEURS_BASE+'/audio/test.tar.gz', timeout=60) as response:
        with tarfile.open(fileobj=response, mode='r|gz') as tar:
            for member in tar:
                name = Path(member.name).name
                meta = references.get(name)
                if not member.isfile() or meta is None or meta['prompt_id'] in used_prompts:
                    continue
                extracted = tar.extractfile(member)
                if extracted is None:
                    continue
                payload = extracted.read()
                wave, native_sr = read_audio(io.BytesIO(payload))
                if not 3 <= len(wave)/48000 <= 15:
                    continue  # Never truncate speech: transcript stays aligned with the full recording.
                wave = wave * (.05/max(float(np.sqrt(np.mean(wave.astype(float)**2))), 1e-10))
                wave *= min(1., .95/max(float(np.max(np.abs(wave))), 1e-10))
                speech.append(dict(meta, wave=wave.astype(np.float32), native_sr=native_sr,
                                   source_file=name, sha256=hashlib.sha256(payload).hexdigest()))
                used_prompts.add(meta['prompt_id'])
                print(f'Selected {len(speech)}/{count}: {name} ({len(wave)/48000:.1f}s, native {native_sr}Hz)', flush=True)
                if len(speech) == count:
                    break
    if len(speech) != count:
        raise RuntimeError(f'Only found {len(speech)} valid unique FLEURS prompts')
    rows = []
    def write_input(identifier, wave, **metadata):
        input_file = f'input_48k/{identifier}.wav'
        raw_file = f'raw_16k/{identifier}.wav'
        sf.write(root/input_file, wave, 48000, subtype='FLOAT')
        sf.write(root/raw_file, resample_poly(wave, 1, 3), 16000, subtype='FLOAT')
        rows.append(dict(id=identifier, input_48k=input_file, raw_16k=raw_file,
                         df3_16k=f'df3_16k/{identifier}.wav', audio_seconds=len(wave)/48000, **metadata))
    for index, sample in enumerate(speech):
        identifier = f'vi_{index:03d}'
        speech_mask = frame_activity(sample['wave'], 48000)
        common = {k: v for k, v in sample.items() if k != 'wave'}
        common['speech_active_intervals'] = activity_intervals(speech_mask, 48000)
        write_input(identifier+'_clean', sample['wave'], kind='clean', snr_db=None,
                    noise_type=None, noise_style='clean', events=[], noise_active_intervals=[],
                    achieved_snr_db=None, mixture_gain=1., **common)
        for noise_type, style in NOISE_CONDITIONS:
            track, mask, events = build_noise_track(banks[noise_type], len(sample['wave']), 48000,
                                                    rng, noise_type, style, speech_mask)
            for snr in (0, 5, 10):
                mixed, _, actual_snr, gain = mix_at_snr(sample['wave'], track, snr,
                                                       active_mask=mask, speech_mask=speech_mask)
                write_input(identifier+f'_{noise_type}_{style}_snr{snr:02d}', mixed,
                            kind='noisy', snr_db=snr, noise_type=noise_type, noise_style=style,
                            events=events, noise_active_intervals=activity_intervals(mask, 48000),
                            achieved_snr_db=actual_snr, mixture_gain=gain, **common)
    for noise_type, style in NOISE_CONDITIONS:
        for index in range(3):
            wave, mask, events = build_noise_track(banks[noise_type], 10*48000, 48000, rng,
                                      noise_type, style, np.zeros(10*48000, dtype=bool))
            wave = wave * (.05/float(np.sqrt(np.mean(wave[mask].astype(float)**2))))
            wave *= min(1., .95/max(float(np.max(np.abs(wave))), 1e-10))
            write_input(f'{noise_type}_{style}_only_{index:02d}', wave, kind='noise_only',
                        snr_db=None, text='', prompt_id=None, noise_type=noise_type, noise_style=style,
                        events=events, noise_active_intervals=activity_intervals(mask, 48000),
                        speech_active_intervals=[], achieved_snr_db=None, mixture_gain=None)
    manifest = dict(schema_version=2, seed=seed, speech_clips=count, snrs_db=[0, 5, 10], samples=rows,
                    noise_conditions=NOISE_CONDITIONS,
                    fleurs_revision=FLEURS_REVISION, fleurs_source=FLEURS_BASE,
                    noise_source=DCASE_URL, noise_md5=DCASE_MD5,
                    noise_recordings=[{k:v for k,v in item.items() if k!='wave'}
                                      for bank in banks.values() for item in bank],
                    licenses={'FLEURS': 'CC BY 4.0', 'DCASE2016 archive': 'CC BY 3.0'},
                    snr_definition='RMS of active speech across utterance / RMS of active noise events; '
                        '20ms energy gates: speech -35dB, noise -45dB relative to each track peak. '
                        'Not a true VAD. Same event positions across SNRs, joint anti-clipping gain.',
                    notes='First unique full 3-15s prompts in archive order. Event onset labels use '
                          'energy activity; events may extend into speech from a pause onset. '
                          'Inspect speech_overlap_fraction. Continuous keyboard is a separate condition.')
    save_json(root/'manifest.json', manifest)
    print('Prepared', len(rows), 'inputs:', count, 'clean,',
          count*len(NOISE_CONDITIONS)*3, 'noisy,', 3*len(NOISE_CONDITIONS), 'noise-only', flush=True)


def enhance_dataset(root):
    import torch
    from df.enhance import init_df, enhance
    from importlib.metadata import version
    if not torch.cuda.is_available():
        raise RuntimeError('DF3 requires CUDA')
    root = Path(root)
    manifest = read_json(root/'manifest.json')
    model, state, _, _ = init_df(model_base_dir='DeepFilterNet3', log_file=None)
    if next(model.parameters()).device.type != 'cuda' or state.sr() != 48000:
        raise RuntimeError('DF3 model must be CUDA / 48kHz')
    model.eval()
    first, _ = read_audio(root/manifest['samples'][0]['input_48k'])
    with torch.inference_mode():
        for _ in range(2):
            enhance(model, state, torch.from_numpy(first).unsqueeze(0), pad=True)
        torch.cuda.synchronize()
        metrics = {}
        for index, sample in enumerate(manifest['samples']):
            wave, _ = read_audio(root/sample['input_48k'])
            tensor = torch.from_numpy(np.ascontiguousarray(wave)).unsqueeze(0)
            torch.cuda.synchronize()
            start = time.perf_counter()
            cleaned = enhance(model, state, tensor, pad=True)
            torch.cuda.synchronize()
            seconds = time.perf_counter()-start
            array = cleaned.squeeze(0).detach().cpu().numpy()
            if array.shape != wave.shape or not np.isfinite(array).all():
                raise RuntimeError('DF3 returned invalid waveform/length')
            start = time.perf_counter()
            output = resample_poly(array, 1, 3).astype(np.float32)
            resample_seconds = time.perf_counter()-start
            sf.write(root/sample['df3_16k'], output, 16000, subtype='FLOAT')
            input_rms = float(np.sqrt(np.mean(wave.astype(float)**2)))
            output_rms = float(np.sqrt(np.mean(array.astype(float)**2)))
            metrics[sample['id']] = dict(enhancement_seconds=seconds, resample_seconds=resample_seconds,
                output_peak=float(np.max(np.abs(output))),
                rms_attenuation_db=20*math.log10(max(input_rms, 1e-12)/max(output_rms, 1e-12)))
            print(f'DF3 {index+1}/{len(manifest["samples"])}: {sample["id"]}', flush=True)
    save_json(root/'enhancement.json', dict(device='cuda', gpu=torch.cuda.get_device_name(0),
              torch_version=torch.__version__, deepfilternet_version=version('deepfilternet'),
              manifest_sha256=hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest(),
              samples=metrics, notes='Two warmups; one timed enhancement/clip, CPU frontend/synthesis included; resampling separate.'))


def recognize_dataset(root, model_dir, tokens, profile_dir):
    import ctypes
    import sherpa_onnx
    from importlib.metadata import version
    # Reuse the exact tested CUDA profile guard from the original worker.
    from asr_worker import inspect_cuda_profiles
    evidence = inspect_cuda_profiles(sorted(Path(profile_dir).glob('sherpa*.json')))
    root, model_dir = Path(root), Path(model_dir)
    manifest = read_json(root/'manifest.json')
    enhancement = read_json(root/'enhancement.json')
    expected = {sample['id'] for sample in manifest['samples']}
    if (set(enhancement['samples']) != expected or enhancement.get('manifest_sha256') !=
            hashlib.sha256((root/'manifest.json').read_bytes()).hexdigest()):
        raise RuntimeError('Enhancement/manifest IDs differ; rerun DF3 for this dataset')
    recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
        tokens=str(tokens), encoder=str(model_dir/'encoder-epoch-20-avg-10.onnx'),
        decoder=str(model_dir/'decoder-epoch-20-avg-10.onnx'),
        joiner=str(model_dir/'joiner-epoch-20-avg-10.onnx'), num_threads=2,
        sample_rate=16000, feature_dim=80, decoding_method='greedy_search', provider='cuda')
    cudart = ctypes.CDLL('libcudart.so.12')
    cudart.cudaDeviceSynchronize.restype = ctypes.c_int
    def recognize(wave):
        if cudart.cudaDeviceSynchronize():
            raise RuntimeError('CUDA synchronization failed')
        start = time.perf_counter()
        stream = recognizer.create_stream()
        stream.accept_waveform(16000, wave)
        recognizer.decode_stream(stream)
        text = stream.result.text
        if cudart.cudaDeviceSynchronize():
            raise RuntimeError('CUDA synchronization failed')
        return text, time.perf_counter()-start
    for _ in range(5):
        for name in ('raw_16k', 'df3_16k'):
            wave, _ = read_audio(root/manifest['samples'][0][name], 16000)
            recognize(wave)
    rows = []
    for index, sample in enumerate(manifest['samples']):
        pair = {}
        for branch in ('raw', 'df3'):
            wave, native_sr = read_audio(root/sample[branch+'_16k'], 16000)
            if native_sr != 16000:
                raise RuntimeError('ASR input must already be 16 kHz')
            pair[branch] = wave
        if len(pair['raw']) != len(pair['df3']):
            raise RuntimeError('Raw/DF3 lengths differ')
        for branch in (('raw', 'df3') if index % 2 == 0 else ('df3', 'raw')):
            text, seconds = recognize(pair[branch])
            row = dict(sample, branch=branch, transcript=text, asr_seconds=seconds,
                       **error_counts(sample['text'], text))
            if branch == 'df3':
                row.update(enhancement['samples'][sample['id']])
            rows.append(row)
        print(f'ASR {index+1}/{len(manifest["samples"])}: {sample["id"]}', flush=True)
    summary = summarize(rows)
    save_json(root/'quality_results.json', dict(samples=rows, summary=summary,
        cuda_probe_executed_node_counts=evidence, sherpa_version=version('sherpa-onnx'),
        model_revision=model_dir.name, df3_gpu=enhancement['gpu'],
        notes='Corpus WER/CER sum edit counts before division; casefold+NFC, punctuation removed, tones retained, digits not verbalized. Vietnamese WER uses whitespace tokens; CER excludes spaces. Empty-reference noise clips excluded from WER/CER, reported by nonempty transcript rate. One ASR run/clip; latency aggregates different clips, not repeated performance runs. Pipeline latency is a sum across warm separate workers, not measured end-to-end.'))
    for name, data in [('quality_per_clip.csv', rows), ('quality_summary.csv', summary)]:
        keys = list(dict.fromkeys(k for row in data for k in row))
        with (root/name).open('w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=keys)
            writer.writeheader()
            writer.writerows(data)
    print(json.dumps(summary, ensure_ascii=False, indent=2), flush=True)


def main():
    p = argparse.ArgumentParser(description=__doc__)
    p.add_argument('mode', choices=['prepare', 'df3', 'asr'])
    p.add_argument('--root', required=True)
    p.add_argument('--count', type=int, default=20)
    p.add_argument('--seed', type=int, default=20261019)
    p.add_argument('--model-dir')
    p.add_argument('--tokens')
    p.add_argument('--profile-dir')
    args = p.parse_args()
    if args.mode == 'prepare':
        prepare(args.root, args.count, args.seed)
    elif args.mode == 'df3':
        enhance_dataset(args.root)
    else:
        if not all((args.model_dir, args.tokens, args.profile_dir)):
            p.error('asr requires --model-dir, --tokens, --profile-dir')
        recognize_dataset(args.root, args.model_dir, args.tokens, args.profile_dir)


if __name__ == '__main__':
    main()
'''

import os, sys, subprocess, json
from pathlib import Path
for required in ('DF_PY', 'model_dir', 'tokens_path', 'asr_env', 'ASR_WORKER_SOURCE'):
    assert required in globals(), f'Thiếu {required}: chạy lần lượt các mục 1–5'
assert Path(DF_PY).exists()
QUALITY_ROOT = Path('/content/zipformer30m_quality')
QUALITY_ROOT.mkdir(exist_ok=True)
quality_worker = Path('/content/quality_benchmark.py')
quality_worker.write_text(QUALITY_WORKER_SOURCE, encoding='utf-8')
# Reuse the verified guard without creating another GPU context in the notebook kernel.
Path('/content/asr_worker.py').write_text(ASR_WORKER_SOURCE)
N_SPEECH, DATA_SEED = 20, 20261019

def run_quality_worker(mode):
    executable = str(DF_PY) if mode == 'df3' else sys.executable
    command = [executable, str(quality_worker), mode, '--root', str(QUALITY_ROOT)]
    if mode == 'prepare':
        command += ['--count', str(N_SPEECH), '--seed', str(DATA_SEED)]
    if mode == 'asr':
        assert 'profile_dir' in globals(), 'Chạy probe CUDA ở mục 8 trước'
        assert list(Path(profile_dir).glob('sherpa*.json')), 'Thiếu CUDA profile'
        command += ['--model-dir', str(model_dir), '--tokens', str(tokens_path),
                    '--profile-dir', str(profile_dir)]
    environment = asr_env.copy() if mode == 'asr' else os.environ.copy()
    environment['CUDA_VISIBLE_DEVICES'] = '0'
    log_path = QUALITY_ROOT/f'{mode}_stderr.log'
    with log_path.open('w') as log:
        result = subprocess.run(command, env=environment, stderr=log)
    diagnostic = log_path.read_text(errors='replace')
    if result.returncode:
        print(diagnostic[-16000:])
        raise RuntimeError(f'Quality {mode} lỗi {result.returncode}; xem {log_path}')
    if mode == 'asr':
        failures = native_cpu_fallback_lines(diagnostic)
        if failures:
            print('\n'.join(failures))
            raise RuntimeError('ASR provider fallback CPU; xem log trước khi dùng kết quả')
    print('Completed:', mode, '| stderr:', log_path)

## 6. Tạo dataset nhiễu ngắt quãng và chọn mẫu benchmark tự động
- Keyboard: các đợt gõ 0,5–2 giây, có khoảng nghỉ; có thêm nhóm liên tục riêng.
- Gõ cửa, đóng cửa, chìa khóa, lật giấy: 1–3 sự kiện, giữ toàn bộ bản ghi và đuôi âm nếu đủ chỗ.
- Chuông: một đợt gồm 2–3 tiếng chuông gốc cách nhau 0,6 giây; nếu không đủ chỗ, dùng một tiếng chuông trọn vẹn.
Các sự kiện cách nhau ít nhất 0,3 giây. Seed cố định; cùng waveform nhiễu và thời điểm giữa ba SNR.
Mỗi sự kiện ghi thời điểm bắt đầu/kết thúc, file nguồn và tỷ lệ chồng lên vùng có tiếng nói ước lượng.

**SNR mới:** RMS tiếng nói trong các frame có tiếng nói / RMS nhiễu trong các frame có nhiễu.
Dùng gate năng lượng RMS 20 ms, không phải VAD hay căn chỉnh theo từ. Điểm bắt đầu được ưu tiên ở speech hoặc pause;
nếu không có vị trí phù hợp sẽ ghi `unconstrained`. Sự kiện bắt đầu trong pause vẫn có thể kéo dài đè lên tiếng nói.
Một gain chung bảo vệ peak ≤0,95; lưu FLOAT32. Các khoảng nghỉ không làm tăng gain nhiễu.

In [ ]:
run_quality_worker('prepare')
quality_manifest = json.loads((QUALITY_ROOT/'manifest.json').read_text())
from collections import Counter
import soundfile as sf
import numpy as np
from scipy.signal import resample_poly
print('Inputs:', Counter(sample['kind'] for sample in quality_manifest['samples']))
print('Conditions:', quality_manifest['noise_conditions'])
PERF_SAMPLE_ID = 'vi_000_keyboard_events_snr05'
perf_sample = next(sample for sample in quality_manifest['samples'] if sample['id'] == PERF_SAMPLE_ID)
raw_48k = '/content/asr_test_raw_48k.wav'
audio_path = '/content/asr_test_16k.wav'
wave_48k, sr_48k = sf.read(QUALITY_ROOT/perf_sample['input_48k'], dtype='float32')
assert sr_48k == 48000
sf.write(raw_48k, wave_48k, 48000, subtype='FLOAT')
raw_16k = resample_poly(wave_48k, up=1, down=3).astype(np.float32)
sf.write(audio_path, raw_16k, 16000, subtype='FLOAT')
print('Latency sample:', PERF_SAMPLE_ID, '| reference:', perf_sample['text'])

Completed: prepare | stderr: /content/zipformer30m_quality/prepare_stderr.log
Inputs: Counter({'noisy': 420, 'noise_only': 21, 'clean': 20})
Conditions: [['keyboard', 'continuous'], ['keyboard', 'events'], ['knock', 'events'], ['doorslam', 'events'], ['keys', 'events'], ['pageturn', 'events'], ['phone', 'events']]
Latency sample: vi_000_keyboard_events_snr05 | reference: Văn hóa và bộ lạc cổ xưa đã bắt đầu giữ những con vật này để dễ lấy sữa, tóc, thịt, và da.


## 7. Đo warm DF3 trên mẫu dataset tự chọn
Model tải một lần, hai warm-up và năm lượt đo enhancement. Không gồm startup, model load, I/O hay resample.
Ở đây `raw` là audio trước khử nhiễu của `PERF_SAMPLE_ID`, không có file upload.

In [ ]:
DF3_WORKER_SOURCE = r'''
# Isolated DeepFilterNet3 Python 3.12.3 worker.
import argparse, json, time, sys
from pathlib import Path
from importlib.metadata import version
import numpy as np
import torch
import soundfile as sf
from scipy.signal import resample_poly
from df.enhance import init_df, enhance

p = argparse.ArgumentParser()
p.add_argument('--input', required=True)
p.add_argument('--output', required=True)
p.add_argument('--metrics', required=True)
a = p.parse_args()
assert sys.version_info[:3] == (3, 12, 3), sys.version
assert torch.cuda.is_available(), 'DF3 worker CUDA unavailable'
load_start = time.perf_counter()
model, state, _, _ = init_df(model_base_dir='DeepFilterNet3', log_file=None)
assert next(model.parameters()).device.type == 'cuda', 'DF3 model is not on CUDA'
assert state.sr() == 48000
model.eval()
torch.cuda.synchronize()
load_seconds = time.perf_counter() - load_start
wave, sr = sf.read(a.input, dtype='float32')
assert sr == 48000 and wave.ndim == 1 and wave.size > 0
assert np.isfinite(wave).all(), 'Audio has NaN/Inf'
wave = torch.from_numpy(np.ascontiguousarray(wave)).unsqueeze(0)

@torch.inference_mode()
def run_once():
    torch.cuda.synchronize()
    start = time.perf_counter()
    out = enhance(model, state, wave, pad=True)
    torch.cuda.synchronize()
    return out, time.perf_counter() - start

for _ in range(2):
    run_once()
times = []
for _ in range(5):
    cleaned, elapsed = run_once()
    times.append(elapsed)
cleaned_array = cleaned.squeeze(0).detach().cpu().numpy()
assert cleaned_array.shape == (wave.shape[-1],), 'DF3 changed audio length'
assert np.isfinite(cleaned_array).all()
start_rs = time.perf_counter()
cleaned_16k = resample_poly(cleaned_array, up=1, down=3).astype(np.float32)
rs_seconds = time.perf_counter() - start_rs
sf.write(a.output, cleaned_16k, 16000, subtype='FLOAT')
audio_seconds = wave.shape[-1] / 48000
metrics = {
    'model': 'DeepFilterNet3', 'device': 'cuda',
    'gpu': torch.cuda.get_device_name(0),
    'torch_version': torch.__version__, 'torch_cuda_version': torch.version.cuda,
    'deepfilternet_version': version('deepfilternet'),
    'source_sr': 48000, 'asr_sr': 16000, 'worker_python': sys.version,
    'model_load_seconds': load_seconds,
    'warmup_runs': 2, 'measured_runs': len(times),
    'enhancement_latencies_seconds': times,
    'median_enhancement_seconds': float(np.median(times)),
    'p95_enhancement_seconds': float(np.percentile(times, 95)),
    'single_resample_seconds': rs_seconds, 'audio_seconds': audio_seconds,
    'enhancement_median_rtf': float(np.median(times)) / audio_seconds,
    'output_peak': float(np.max(np.abs(cleaned_16k))),
    'output_fraction_above_one': float(np.mean(np.abs(cleaned_16k) > 1)),
    'notes': 'Warm enhancement includes CPU STFT/synthesis, transfers and CUDA network; excludes startup/model load/I/O/resampling. Resample timed once.'}
Path(a.metrics).write_text(json.dumps(metrics, indent=2))
print('DF3:', metrics['gpu'], '| median:', metrics['median_enhancement_seconds'], flush=True)
'''


import json, subprocess
from pathlib import Path
worker_path = Path('/content/df3_worker.py')
worker_path.write_text(DF3_WORKER_SOURCE)
subprocess.run([str(DF_PY), str(worker_path), '--input', raw_48k,
                '--output', '/content/asr_test_df3_16k.wav',
                '--metrics', '/content/df3_metrics.json'], check=True)
DF3_METRICS = json.loads(Path('/content/df3_metrics.json').read_text())
print(json.dumps(DF3_METRICS, indent=2, ensure_ascii=False))

{
  "model": "DeepFilterNet3",
  "device": "cuda",
  "gpu": "Tesla T4",
  "torch_version": "2.5.1+cu121",
  "torch_cuda_version": "12.1",
  "deepfilternet_version": "0.5.7rc0",
  "source_sr": 48000,
  "asr_sr": 16000,
  "worker_python": "3.12.3 (main, Apr 15 2024, 18:25:56) [Clang 17.0.6 ]",
  "model_load_seconds": 1.7457246929999997,
  "warmup_runs": 2,
  "measured_runs": 5,
  "enhancement_latencies_seconds": [
    0.09000340100004678,
    0.08309405400001424,
    0.07197187399998484,
    0.07638106600001038,
    0.06985811199996306
  ],
  "median_enhancement_seconds": 0.07638106600001038,
  "p95_enhancement_seconds": 0.08862153160004027,
  "single_resample_seconds": 0.014702008999961436,
  "audio_seconds": 11.34,
  "enhancement_median_rtf": 0.006735543738977987,
  "output_peak": 0.4978387653827667,
  "output_fraction_above_one": 0.0,
  "notes": "Warm enhancement includes CPU STFT/synthesis, transfers and CUDA network; excludes startup/model load/I/O/resampling. Resample timed once."


## 8. Probe bắt buộc có node CUDA thực thi
Profiling của runtime bên trong Sherpa phải có event Node/CUDAExecutionProvider.
Phân bổ từng node sang CPU là hợp lệ; fallback provider vẫn bị chặn. VRAM chỉ là số liệu hỗ trợ.

In [ ]:
import tempfile, json, subprocess
from pathlib import Path
profile_dir = Path(tempfile.mkdtemp(prefix='zipformer_cuda_probe_', dir='/content'))
common_args = [sys.executable, str(asr_worker), '--model-dir', str(model_dir),
               '--tokens', str(tokens_path), '--profile-dir', str(profile_dir)]

def native_cpu_fallback_lines(diagnostic):
    failures = []
    for line in diagnostic.splitlines():
        lower = line.lower()
        if 'fallback to cpu' not in lower and 'fall back to cpu' not in lower:
            continue
        # ORT deliberately places some individual nodes on CPU even with CUDA enabled.
        cpu_node_placement = ('[i:onnxruntime:' in lower
                              and 'fallback_cpu_capability.cc:' in lower)
        if not cpu_node_placement:
            failures.append(line)
    return failures


def run_asr_worker(mode):
    log_path = Path(f'/content/asr_{mode}_stderr.log')
    with log_path.open('w') as log:
        result = subprocess.run(common_args + ['--mode', mode], env=asr_env, stderr=log)
    diagnostic = log_path.read_text(errors='replace')
    if result.returncode:
        print(diagnostic[-16000:])
        raise RuntimeError(f'ASR worker {mode} lỗi {result.returncode}; xem {log_path}')
    failures = native_cpu_fallback_lines(diagnostic)
    if failures:
        print('Provider fallback diagnostics:\n' + '\n'.join(failures))
        raise RuntimeError(f'Sherpa báo provider fallback CPU; xem {log_path}: {failures[0]}')
    print('Native stderr:', log_path)

run_asr_worker('probe')
# Load only the stdlib CUDA-evidence parser, without importing GPU packages into the kernel.
import ast
from collections import Counter
node = next(n for n in ast.parse(ASR_WORKER_SOURCE).body
            if isinstance(n, ast.FunctionDef) and n.name == 'inspect_cuda_profiles')
exec(compile(ast.Module(body=[node], type_ignores=[]), '<profile-check>', 'exec'))
CUDA_EVIDENCE = inspect_cuda_profiles(sorted(profile_dir.glob('sherpa*.json')))
print('Executed node events:', CUDA_EVIDENCE)
print('Profiles:', profile_dir)

Native stderr: /content/asr_probe_stderr.log
Executed node events: {'CUDAExecutionProvider': 3493, 'CPUExecutionProvider': 1139}
Profiles: /content/zipformer_cuda_probe_98jf2ie8


## 9. Benchmark ASR lặp trên mẫu tự chọn
5 warm-up/nhánh và 20 lượt đo/nhánh, đảo thứ tự raw/DF3. NVML bật từ trước warm-up.
Thời gian gồm stream + waveform + decode + lấy text + CUDA sync; không gồm file I/O và model load.

In [ ]:
run_asr_worker('benchmark')
reports = {}
for variant in ('raw', 'df3'):
    path = Path(f'/content/zipformer30m_{variant}_benchmark.json')
    reports[variant] = json.loads(path.read_text())
    report = reports[variant]
    print(f"{variant}: median {report['median_latency_seconds']*1000:.2f} ms | "
          f"P95 {report['p95_latency_seconds']*1000:.2f} ms | RTF {report['median_rtf']:.5f}")
    print('First 5:', report['latencies_seconds'][:5])
    print('Memory peak total/PID:', report['gpu_memory_peak_total_mib'], report['gpu_memory_peak_process_mib'])
    print('Monitor errors:', report['memory_monitor_errors'])

Native stderr: /content/asr_benchmark_stderr.log
raw: median 91.87 ms | P95 95.35 ms | RTF 0.00810
First 5: [0.09532205300001806, 0.09342484500001547, 0.09264854400004197, 0.09092487399999527, 0.09451693400001204]
Memory peak total/PID: 890.1875 440.0
Monitor errors: []
df3: median 94.29 ms | P95 99.00 ms | RTF 0.00832
First 5: [0.09791248499999483, 0.09538596400000188, 0.09560556299999234, 0.09465492999999015, 0.09418711399996482]
Memory peak total/PID: 890.1875 440.0
Monitor errors: []


## 10. Khử nhiễu cả dataset trong một worker CUDA
Bao gồm câu sạch để kiểm tra DF3 có làm tăng lỗi. Mỗi clip đo một lượt sau hai warm-up.
Giữ kết quả latency lặp ở mục 9 để so sánh hiệu năng; các clip ở đây có độ dài khác nhau.

In [ ]:
run_quality_worker('df3')

Completed: df3 | stderr: /content/zipformer30m_quality/df3_stderr.log


## 11. Nhận dạng và chấm riêng từng điều kiện
WER/CER tính theo tổng edit count / tổng reference count, không lấy trung bình tỷ lệ từng câu.
Giữ dấu tiếng Việt, NFC + chữ thường, bỏ dấu câu; số không tự đọc thành chữ.
WER tiếng Việt dùng token cách nhau bằng khoảng trắng, CER bỏ khoảng trắng.
Noise-only không có WER/CER, thay bằng tỷ lệ sinh transcript không rỗng trên 3 mẫu mỗi điều kiện.

In [ ]:
run_quality_worker('asr')
quality_results = json.loads((QUALITY_ROOT/'quality_results.json').read_text())
from IPython.display import display, Markdown
summary = quality_results['summary']
lines = ['| Loại | Nhiễu | Chế độ | SNR | WER raw | WER DF3 | CER raw | CER DF3 | ΔWER (điểm %) |',
         '|---|---|---|---:|---:|---:|---:|---:|---:|']
keys = list(dict.fromkeys((r['kind'], r['noise_type'], r['noise_style'], r['snr_db'])
                         for r in summary if r['kind'] != 'noise_only'))
for kind, noise_type, style, snr in keys:
    pair = {r['branch']: r for r in summary if
            (r['kind'], r['noise_type'], r['noise_style'], r['snr_db']) == (kind, noise_type, style, snr)}
    raw, cleaned = pair['raw'], pair['df3']
    lines.append(f"| {kind} | {noise_type or '—'} | {style} | {'—' if snr is None else snr} | "
                 f"{raw['wer']:.2%} | {cleaned['wer']:.2%} | {raw['cer']:.2%} | {cleaned['cer']:.2%} | "
                 f"{(cleaned['wer']-raw['wer'])*100:+.2f} |")
display(Markdown('\n'.join(lines)))
print('ΔWER âm = giảm lỗi; ΔWER dương = tăng lỗi.')
for r in summary:
    if r['kind'] == 'noise_only':
        print(f"Noise-only / {r['noise_type']} / {r['noise_style']} / {r['branch']}: "
              f"{r['nonempty_transcript_rate']:.0%} sinh chữ ({r['clips']} clip)")

Completed: asr | stderr: /content/zipformer30m_quality/asr_stderr.log


| Loại | Nhiễu | Chế độ | SNR | WER raw | WER DF3 | CER raw | CER DF3 | ΔWER (điểm %) |
|---|---|---|---:|---:|---:|---:|---:|---:|
| clean | — | clean | — | 8.52% | 8.15% | 5.05% | 4.94% | -0.37 |
| noisy | keyboard | continuous | 0 | 9.26% | 10.19% | 6.19% | 6.78% | +0.93 |
| noisy | keyboard | continuous | 5 | 9.26% | 9.63% | 5.70% | 6.19% | +0.37 |
| noisy | keyboard | continuous | 10 | 8.52% | 8.89% | 5.32% | 5.48% | +0.37 |
| noisy | keyboard | events | 0 | 8.89% | 9.81% | 5.70% | 6.46% | +0.93 |
| noisy | keyboard | events | 5 | 8.89% | 9.81% | 5.37% | 6.24% | +0.93 |
| noisy | keyboard | events | 10 | 9.26% | 8.89% | 5.32% | 6.08% | -0.37 |
| noisy | knock | events | 0 | 8.52% | 8.70% | 4.99% | 5.43% | +0.19 |
| noisy | knock | events | 5 | 8.52% | 8.70% | 5.05% | 5.59% | +0.19 |
| noisy | knock | events | 10 | 8.52% | 8.70% | 5.05% | 5.53% | +0.19 |
| noisy | doorslam | events | 0 | 8.70% | 8.89% | 5.21% | 5.43% | +0.19 |
| noisy | doorslam | events | 5 | 8.70% | 8.70% | 5.21% | 5.10% | +0.00 |
| noisy | doorslam | events | 10 | 8.70% | 8.70% | 5.15% | 5.10% | +0.00 |
| noisy | keys | events | 0 | 8.52% | 8.70% | 4.99% | 5.64% | +0.19 |
| noisy | keys | events | 5 | 8.52% | 8.52% | 5.05% | 5.64% | +0.00 |
| noisy | keys | events | 10 | 8.52% | 8.33% | 4.99% | 5.21% | -0.19 |
| noisy | pageturn | events | 0 | 9.26% | 10.19% | 5.86% | 6.95% | +0.93 |
| noisy | pageturn | events | 5 | 8.89% | 9.44% | 5.32% | 6.35% | +0.56 |
| noisy | pageturn | events | 10 | 8.89% | 9.07% | 5.26% | 5.48% | +0.19 |
| noisy | phone | events | 0 | 8.70% | 10.19% | 5.21% | 7.05% | +1.48 |
| noisy | phone | events | 5 | 8.70% | 9.07% | 5.15% | 5.53% | +0.37 |
| noisy | phone | events | 10 | 8.70% | 9.26% | 5.15% | 5.97% | +0.56 |

ΔWER âm = giảm lỗi; ΔWER dương = tăng lỗi.
Noise-only / keyboard / continuous / raw: 0% sinh chữ (3 clip)
Noise-only / keyboard / continuous / df3: 0% sinh chữ (3 clip)
Noise-only / keyboard / events / df3: 33% sinh chữ (3 clip)
Noise-only / keyboard / events / raw: 0% sinh chữ (3 clip)
Noise-only / knock / events / raw: 0% sinh chữ (3 clip)
Noise-only / knock / events / df3: 0% sinh chữ (3 clip)
Noise-only / doorslam / events / df3: 0% sinh chữ (3 clip)
Noise-only / doorslam / events / raw: 0% sinh chữ (3 clip)
Noise-only / keys / events / raw: 0% sinh chữ (3 clip)
Noise-only / keys / events / df3: 0% sinh chữ (3 clip)
Noise-only / pageturn / events / df3: 0% sinh chữ (3 clip)
Noise-only / pageturn / events / raw: 33% sinh chữ (3 clip)
Noise-only / phone / events / raw: 67% sinh chữ (3 clip)
Noise-only / phone / events / df3: 100% sinh chữ (3 clip)


## 12. Nghe và kiểm tra vị trí sự kiện
Chọn ID bằng `LISTEN_ID` trong manifest hoặc CSV, ví dụ `vi_000_doorslam_events_snr05`,
`vi_000_keyboard_continuous_snr00`, `phone_events_only_00`.
Audio dùng cùng gain playback cho hai nhánh, không tự nâng tiếng nhỏ sau DF3.
Các timestamp được in bên dưới giúp đối chiếu khoảng nghỉ và sự kiện; `speech_overlap_fraction` là ước lượng theo năng lượng.

In [ ]:
from IPython.display import Audio, display
import soundfile as sf
import numpy as np
import ipywidgets as widgets
from collections import Counter
available = [row for row in quality_results['samples'] if row['branch'] == 'raw']
print('Số đầu vào thực tế:', len(available))
for condition, count in Counter((r['kind'], r['noise_type'], r['noise_style']) for r in available).items():
    print(condition, ':', count)
selector = widgets.Dropdown(
    options=[(row['id'], row['id']) for row in available],
    value=PERF_SAMPLE_ID, description='Mẫu audio:', layout=widgets.Layout(width='95%'))

def play_sample(sample_id):
    global LISTEN_ID
    LISTEN_ID = sample_id
    pair = {row['branch']: row for row in quality_results['samples'] if row['id'] == LISTEN_ID}
    assert len(pair) == 2, 'Chọn LISTEN_ID từ quality_per_clip.csv'
    print('Reference:', pair['raw']['text'] or '(chỉ có nhiễu)')
    for event in pair['raw']['events']:
        print(f"Event {event['start_sample']/48000:.2f}–{event['end_sample']/48000:.2f}s | "
              f"{event['source_file']} | onset={event['onset_region']} | "
              f"speech overlap={event['speech_overlap_fraction']:.0%}")
    playback = {branch: sf.read(QUALITY_ROOT/pair[branch][branch+'_16k'], dtype='float32')
                for branch in ('raw', 'df3')}
    shared_gain = 1/max(1., *(float(np.max(np.abs(audio))) for audio, sr in playback.values()))
    for branch in ('raw', 'df3'):
        row = pair[branch]
        print(branch, '| Transcript:', row['transcript'] or '(rỗng)', '| WER:', row['wer'])
        audio, sr = playback[branch]
        assert sr == 16000
        display(Audio(audio*shared_gain, rate=sr, normalize=False))

player = widgets.interactive_output(play_sample, {'sample_id': selector})
display(selector, player)

Số đầu vào thực tế: 461
('clean', None, 'clean') : 20
('noisy', 'keyboard', 'continuous') : 60
('noisy', 'keyboard', 'events') : 60
('noisy', 'knock', 'events') : 60
('noisy', 'doorslam', 'events') : 60
('noisy', 'keys', 'events') : 60
('noisy', 'pageturn', 'events') : 60
('noisy', 'phone', 'events') : 60
('noise_only', 'keyboard', 'continuous') : 3
('noise_only', 'keyboard', 'events') : 3
('noise_only', 'knock', 'events') : 3
('noise_only', 'doorslam', 'events') : 3
('noise_only', 'keys', 'events') : 3
('noise_only', 'pageturn', 'events') : 3
('noise_only', 'phone', 'events') : 3


Dropdown(description='Mẫu audio:', index=5, layout=Layout(width='95%'), options=(('vi_000_clean', 'vi_000_clea…

Output()

## 13. Tải kết quả
ZIP mặc định gồm báo cáo, manifest có lịch sự kiện, log, CUDA profile, worker và cặp audio đang nghe.
Đổi `INCLUDE_ALL_AUDIO=True` để thêm toàn bộ 922 audio raw/enhanced; ZIP sẽ lớn hơn đáng kể.
Tổng enhancement + resample + ASR chỉ là ước lượng từ các worker warm riêng, không phải phép đo end-to-end trực tiếp.
NVML lấy mẫu có thể bỏ sót đỉnh ngắn; PID memory gồm context và allocator.

In [ ]:
import zipfile
from google.colab import files
INCLUDE_ALL_AUDIO = True
quality_bundle = Path('/content/zipformer30m_quality_results.zip')
with zipfile.ZipFile(quality_bundle, 'w', compression=zipfile.ZIP_DEFLATED) as z:
    for name in ('manifest.json', 'enhancement.json', 'quality_results.json',
                 'quality_per_clip.csv', 'quality_summary.csv',
                 'prepare_stderr.log', 'df3_stderr.log', 'asr_stderr.log'):
        path = QUALITY_ROOT/name
        z.write(path, arcname=name)
    z.write(quality_worker, arcname='quality_benchmark.py')
    z.write('/content/asr_worker.py', arcname='asr_worker.py')
    for profile in Path(profile_dir).glob('*'):
        if profile.is_file():
            z.write(profile, arcname=f'cuda_probe/{profile.name}')
    for variant in ('raw', 'df3'):
        for ext in ('csv', 'json'):
            path = Path(f'/content/zipformer30m_{variant}_benchmark.{ext}')
            z.write(path, arcname=f'performance/{path.name}')
    z.write('/content/df3_metrics.json', arcname='performance/df3_metrics.json')
    for folder in ('raw_16k', 'df3_16k'):
        for sample in quality_results['samples']:
            if sample['branch'] != ('raw' if folder == 'raw_16k' else 'df3'):
                continue
            if not INCLUDE_ALL_AUDIO and sample['id'] != LISTEN_ID:
                continue
            path = QUALITY_ROOT/sample[folder]
            z.write(path, arcname=str(path.relative_to(QUALITY_ROOT)))
with zipfile.ZipFile(quality_bundle) as check:
    audio_count = sum(name.endswith('.wav') for name in check.namelist())
expected_audio_count = len(quality_results['samples']) if INCLUDE_ALL_AUDIO else 2
assert audio_count == expected_audio_count, (audio_count, expected_audio_count)
print('Số WAV trong ZIP:', audio_count)
print('ZIP:', quality_bundle, f'({quality_bundle.stat().st_size/2**20:.1f} MiB)')
files.download(str(quality_bundle))

Số WAV trong ZIP: 922
ZIP: /content/zipformer30m_quality_results.zip (566.6 MiB)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>